- its also called ReAct Pattern (Reason + Act)
- The LLM thinks, decides an action, observes result, then repeats.

`Thought → Action → Observation → Thought → Action → Observation`

EX: Find top selling Amazon products and create a report

Thought: Need Amazon data
Action: Call Amazon API

Observation: Got data

Thought: Need summary
Action: Generate report



```
User Question
      ↓
LLM (with REACT_SYSTEM_PROMPT)
      ↓
Tool Call ?
      ↓
Execute Tool
      ↓
Observation
      ↓
LLM again
      ↓
Repeat until <response>
```

In [34]:
import json
import re

from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage, AIMessage

In [35]:
# Replace with your OpenAI key
llm = ChatOpenAI(
    model="gpt-4o-mini",   # fast + cheap
    temperature=0.2
)

In [36]:
REACT_SYSTEM_PROMPT = """
You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.

You are provided with function signatures within <tools></tools> XML tags.

You may call one or more functions to assist with the user query.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>, "arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools>
%s
</tools>

If the user asks something unrelated to the tools, respond normally using:

<response>Your answer</response>
"""

In [37]:
# Example tool
def get_current_weather(location: str, unit: str = "celsius"):
    """
    Get current weather for a city
    """

    fake_data = {
        "Madrid": {"temperature": 25, "unit": unit},
        "Delhi": {"temperature": 32, "unit": unit},
        "London": {"temperature": 18, "unit": unit}
    }

    return fake_data.get(location, {"error": "location not found"})

In [38]:
TOOLS = {
    "get_current_weather": get_current_weather
}

In [39]:
def format_tools(tools):

    tool_list = []

    for name, func in tools.items():
        tool_list.append({
            "name": name,
            "description": func.__doc__,
            "parameters": {
                "location": "string",
                "unit": "string"
            }
        })

    return json.dumps(tool_list, indent=2)

In [40]:
def extract_tool_call(text):

    match = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)

    if not match:
        return None

    return json.loads(match.group(1))

In [41]:
def extract_response(text):

    match = re.search(r"<response>(.*?)</response>", text, re.DOTALL)

    if match:
        return match.group(1)

    return None

In [42]:
def run_tool(tool_call):

    tool_name = tool_call["name"]
    args = tool_call["arguments"]

    if tool_name not in TOOLS:
        return {"error": "tool not found"}

    tool = TOOLS[tool_name]

    try:
        result = tool(**args)
        return result
    except Exception as e:
        return {"error": str(e)}

In [ ]:
def react_agent(question, max_steps=10):

    system_prompt = REACT_SYSTEM_PROMPT % format_tools(TOOLS)

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"<question>{question}</question>")
    ]

    for step in range(max_steps):

        print(f"\n===== STEP {step+1} ==but ===")

        response = llm.invoke(messages)

        output = response.content

        print(output)

        # Check final response
        final_answer = extract_response(output)

        if final_answer:
            return final_answer

        # Extract tool call
        tool_call = extract_tool_call(output)

        if tool_call:

            print("\nRunning tool:", tool_call["name"])

            result = run_tool(tool_call)

            observation = f"<observation>{{{tool_call['id']}: {result}}}</observation>"

            print("Observation:", observation)

            messages.append(AIMessage(content=output))
            messages.append(HumanMessage(content=observation))

        else:
            return "No tool call or final response produced."

    return "Max steps reached."

In [44]:
question = "What is the temperature in Madrid?"

answer = react_agent(question)

print("\nFINAL ANSWER:")
print(answer)


===== STEP 1 =====
<tool_call>
{"name": "get_current_weather", "arguments": {"location": "Madrid", "unit": "metric"}, "id": 1}
</tool_call>

Running tool: get_current_weather
Observation: <observation>{1: {'temperature': 25, 'unit': 'metric'}}</observation>

===== STEP 2 =====
<response>The current temperature in Madrid is 25°C.</response>

FINAL ANSWER:
The current temperature in Madrid is 25°C.
